# Explore NFL Tracking Data Features

This notebook explores the input features for the transformer model and helps identify potential new features to add.

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('darkgrid')
%matplotlib inline

## Mount Google Drive (Colab)

If running on Colab, mount your Google Drive to access the cached data.

In [ ]:
# Mount Google Drive (only needed in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✓ Google Drive mounted")
except:
    IN_COLAB = False
    print("Not running in Colab, skipping Drive mount")

## 1. Load Sample Data

Load data from Google Drive cache or local directory.

In [ ]:
# Try multiple data locations
data_locations = [
    # Google Drive cache (full 18 weeks)
    Path('/content/drive/MyDrive/ExtraDataSportsTrackingTransformer_cache_full18weeks'),
    # Local cache (full 18 weeks)
    Path('data/split_prepped_data_extra_full18weeks'),
    # Google Drive cache (2 weeks)
    Path('/content/drive/MyDrive/ExtraDataSportsTrackingTransformer_cache'),
    # Local cache (2 weeks)
    Path('data/split_prepped_data_extra'),
]

data_dir = None
for loc in data_locations:
    if loc.exists() and (loc / 'train_features.parquet').exists():
        data_dir = loc
        print(f"✓ Found data at: {data_dir}")
        break

if data_dir is None:
    raise FileNotFoundError(
        f"Could not find data in any of these locations:\n" + 
        "\n".join([f"  - {loc}" for loc in data_locations])
    )

# Check what files are available
print(f"\nAvailable files:")
for f in sorted(data_dir.glob('*.parquet')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:30s} ({size_mb:,.1f} MB)")

In [ ]:
# Load features and targets (sample first 100k rows for speed)
print("Loading data...")
features_df = pl.read_parquet(data_dir / 'train_features.parquet').head(100_000)
targets_df = pl.read_parquet(data_dir / 'train_targets.parquet')

print(f"✓ Loaded {len(features_df):,} rows")
print(f"\nColumns ({len(features_df.columns)}): {', '.join(features_df.columns[:10])}...")

## 2. Current Model Features

The transformer model currently uses these 6 features per player:
- `x_rel`: Relative x position (forward/backward from ball)
- `y_rel`: Relative y position (left/right from ball)
- `vx`: Velocity in x direction
- `vy`: Velocity in y direction
- `side`: 1 for offense, -1 for defense
- `is_ball_carrier`: 1 if ball carrier, 0 otherwise

In [ ]:
# Get current model features
model_features = ['x_rel', 'y_rel', 'vx', 'vy', 'side', 'is_ball_carrier']

# Summary statistics for model features
print("Current Model Features:")
print("=" * 60)
for feat in model_features:
    vals = features_df[feat].to_numpy()
    print(f"\n{feat}:")
    print(f"  min={vals.min():.2f}, max={vals.max():.2f}")
    print(f"  mean={vals.mean():.2f}, std={vals.std():.2f}")
    print(f"  nulls={features_df[feat].null_count():,}")

## 3. Available Additional Features

Explore what other features are available in the data that we could add to the model.

In [ ]:
# Show all available columns
print("All Available Columns:")
print("=" * 60)
for i, col in enumerate(features_df.columns, 1):
    dtype = features_df[col].dtype
    nulls = features_df[col].null_count()
    used = "✓" if col in model_features else " "
    print(f"{used} {i:2d}. {col:30s} ({dtype})  nulls: {nulls:,}")

## 4. Potential New Features to Add

Let's explore features that might improve the model:
- **Acceleration**: `accel`, `accel_x`, `accel_y`
- **Orientation**: `o`, `dir`, `ox`, `oy`
- **Speed**: `s`
- **Game context**: `down`, `yardsToGo`, `quarter`
- **Position info**: `position`, `position_group`

In [ ]:
# Explore acceleration features
accel_features = ['accel', 'accel_x', 'accel_y']

print("Acceleration Features:")
print("=" * 60)
for feat in accel_features:
    if feat in features_df.columns:
        vals = features_df[feat].to_numpy()
        print(f"\n{feat}:")
        print(f"  min={vals.min():.2f}, max={vals.max():.2f}")
        print(f"  mean={vals.mean():.2f}, std={vals.std():.2f}")
    else:
        print(f"\n{feat}: NOT AVAILABLE")

In [ ]:
# Explore orientation features
orient_features = ['o', 'dir', 'ox', 'oy', 's']

print("Orientation & Speed Features:")
print("=" * 60)
for feat in orient_features:
    if feat in features_df.columns:
        vals = features_df[feat].to_numpy()
        print(f"\n{feat}:")
        print(f"  min={vals.min():.2f}, max={vals.max():.2f}")
        print(f"  mean={vals.mean():.2f}, std={vals.std():.2f}")
    else:
        print(f"\n{feat}: NOT AVAILABLE")

## 5. Visualize Feature Distributions

In [ ]:
# Plot current model features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feat in enumerate(model_features):
    vals = features_df[feat].to_numpy()
    axes[i].hist(vals, bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{feat}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Current Model Features Distribution', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 6. Sample Single Play

Look at all features for a single play to understand the data structure.

In [ ]:
# Get a single play
sample_play = features_df.filter(
    (pl.col('gameId') == features_df['gameId'][0]) &
    (pl.col('playId') == features_df['playId'][0]) &
    (pl.col('frameId') == features_df['frameId'][0])
)

print(f"Sample Play (22 players at one frame):")
print(f"Game: {sample_play['gameId'][0]}, Play: {sample_play['playId'][0]}, Frame: {sample_play['frameId'][0]}")
print(f"\n{sample_play.select(model_features + ['position', 'displayName'])}")

## 7. Feature Correlation Analysis

See which features correlate with yards gained.

In [ ]:
# Join with targets to get yards_gained
df_with_target = features_df.join(
    targets_df,
    on=['gameId', 'playId', 'mirrored', 'frameId'],
    how='inner'
)

print(f"Joined {len(df_with_target):,} rows with targets")

# For numerical features, compute correlation with yards_gained
numerical_features = ['x_rel', 'y_rel', 'vx', 'vy', 's', 'accel', 'o', 'dir', 'accel_x', 'accel_y', 'ox', 'oy']
correlations = {}

for feat in numerical_features:
    if feat in df_with_target.columns:
        # Convert to pandas for correlation calculation
        corr = df_with_target.select([feat, 'yards_gained']).to_pandas().corr().iloc[0, 1]
        correlations[feat] = corr

# Sort by absolute correlation
sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)

print("\nFeature Correlation with Yards Gained:")
print("=" * 60)
for feat, corr in sorted_corr:
    used = "✓" if feat in model_features else " "
    print(f"{used} {feat:20s}: {corr:+.4f}")

## 12. Implementation Recommendations

Based on the exploration above, consider adding these features to improve the model:

### **Priority 1: Game State Features (Recommended)**
These provide critical situational context that significantly affects play outcomes:
- **`yardsToGo`**: Distance needed for first down (1-99 yards, typically 1-20)
- **`down`**: Current down (1-4)
- **`distanceToGoal`**: Distance to opponent's goal line (0-100 yards)
- **`quarter`**: Quarter of game (1-4)
- **`half_seconds_remaining`**: Seconds remaining in current half (0-1800)

**Why these matter**: Teams behave completely differently on 3rd-and-1 vs 1st-and-10, or near the goal line vs midfield. Time pressure in the final minutes of a half drastically changes play calling. These features are:
- Constant across all 22 players (broadcast to each player's embedding)
- Low dimensional (5 features total)
- Highly interpretable
- Capture strategic context not visible in player positions alone

### **Priority 2: Player Motion Features**
Enhance understanding of player movement beyond just velocity:
- **`s`**: Speed magnitude (total velocity)
- **`ox`, `oy`**: Orientation unit vectors (where player is facing)
- **Direction (`dir`)**: Direction of motion

**Implementation approaches:**
1. **For game state features**: Add them as additional input features, broadcasting them to all players
2. **For motion features**: Append to existing per-player feature vector

**Next steps:**
1. Run this notebook to see which features are actually available in your data
2. Check correlations to prioritize features (see Section 10)
3. Modify the model architecture to accept additional features
4. Retrain and compare performance

## 11. Visualize Game State Impact

Create plots showing how game state affects yards gained.

In [ ]:
# QUICK TEST: Game State Feature Correlations
# Copy/paste this cell to quickly test on your data

import polars as pl
from pathlib import Path

# Load data
data_locations = [
    Path('/content/drive/MyDrive/ExtraDataSportsTrackingTransformer_cache_full18weeks'),
    Path('data/split_prepped_data_extra_full18weeks'),
    Path('/content/drive/MyDrive/ExtraDataSportsTrackingTransformer_cache'),
    Path('data/split_prepped_data_extra'),
]

data_dir = None
for loc in data_locations:
    if loc.exists() and (loc / 'train_features.parquet').exists():
        data_dir = loc
        break

if data_dir is None:
    print("❌ No data found!")
else:
    print(f"✓ Loading data from: {data_dir}")
    
    # Load sample (100k rows for speed)
    features_df = pl.read_parquet(data_dir / 'train_features.parquet').head(100_000)
    targets_df = pl.read_parquet(data_dir / 'train_targets.parquet')
    
    # Join features with targets
    df_with_target = features_df.join(
        targets_df,
        on=['gameId', 'playId', 'mirrored', 'frameId'],
        how='inner'
    )
    
    # Get unique frames (game state is constant per frame)
    game_state_features = ['yardsToGo', 'down', 'distanceToGoal', 'quarter', 'half_seconds_remaining']
    available_features = [f for f in game_state_features if f in df_with_target.columns]
    
    unique_frames = df_with_target.select(
        ['gameId', 'playId', 'mirrored', 'frameId', 'yards_gained'] + available_features
    ).unique()
    
    print(f"\n✓ Analyzing {len(unique_frames):,} unique frames")
    print(f"✓ Found {len(available_features)}/{len(game_state_features)} game state features\n")
    
    # Compute correlations
    print("Game State Feature Correlations with Yards Gained:")
    print("=" * 70)
    
    correlations = {}
    for feat in available_features:
        try:
            valid_data = unique_frames.select([feat, 'yards_gained']).drop_nulls()
            if len(valid_data) > 10:  # Need at least 10 samples
                corr = valid_data.to_pandas().corr().iloc[0, 1]
                correlations[feat] = corr
                status = "✓" if abs(corr) > 0.05 else " "
                print(f"{status} {feat:30s}: {corr:+.4f}")
            else:
                print(f"  {feat:30s}: INSUFFICIENT DATA")
        except Exception as e:
            print(f"  {feat:30s}: ERROR - {e}")
    
    # Missing features
    missing = [f for f in game_state_features if f not in available_features]
    if missing:
        print(f"\n❌ Missing features: {', '.join(missing)}")
    
    # Summary
    if correlations:
        print(f"\n✓ Successfully computed {len(correlations)} correlations")
        strongest = max(correlations.items(), key=lambda x: abs(x[1]))
        print(f"✓ Strongest correlation: {strongest[0]} ({strongest[1]:+.4f})")
    else:
        print("\n❌ No correlations computed")

## Quick Test: Game State Feature Correlations

Run this cell to quickly test correlations of all game state features with yards gained.
Copy/paste this cell to test on your data.

## 10. Game State Feature Correlations

Analyze how game state correlates with yards gained.

In [ ]:
# Check which game state features are available
game_state_features = ['yardsToGo', 'quarter', 'down', 'distanceToGoal', 'gameClock', 'half_seconds_remaining']

print("Game State Features:")
print("=" * 60)
for feat in game_state_features:
    if feat in features_df.columns:
        vals = features_df[feat].drop_nulls()
        if len(vals) > 0:
            print(f"\n{feat}: AVAILABLE")
            print(f"  min={vals.min()}, max={vals.max()}")
            if feat not in ['gameClock']:  # Skip mean for time strings
                try:
                    print(f"  mean={vals.mean():.2f}, std={vals.std():.2f}")
                except:
                    print(f"  unique values: {vals.n_unique()}")
            print(f"  nulls={features_df[feat].null_count():,} ({features_df[feat].null_count()/len(features_df)*100:.1f}%)")
        else:
            print(f"\n{feat}: ALL NULL")
    else:
        print(f"\n{feat}: NOT AVAILABLE")

# Show sample values from one play to understand the data
print("\n" + "=" * 60)
print("Sample Play Game State:")
print("=" * 60)
sample_cols = [c for c in game_state_features if c in features_df.columns]
if sample_cols:
    sample_data = sample_play.select(['displayName'] + sample_cols)
    print(sample_data)

In [ ]:
# Plot yards gained by game state features
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

# Use unique frames for plotting
plot_idx = 0

# 1. Yards to Go
if 'yardsToGo' in unique_frames.columns:
    ax = axes[plot_idx]
    plot_idx += 1
    
    # Bin yards to go for clearer visualization
    binned = unique_frames.with_columns(
        yardsToGo_bin=pl.when(pl.col('yardsToGo') <= 3).then(pl.lit('Short (1-3)'))
        .when(pl.col('yardsToGo') <= 7).then(pl.lit('Medium (4-7)'))
        .when(pl.col('yardsToGo') <= 10).then(pl.lit('Long (8-10)'))
        .otherwise(pl.lit('Very Long (11+)'))
    ).drop_nulls()
    
    # Box plot
    data_for_plot = binned.to_pandas()
    data_for_plot.boxplot(column='yards_gained', by='yardsToGo_bin', ax=ax)
    ax.set_title('Yards Gained by Yards To Go')
    ax.set_xlabel('Yards To Go')
    ax.set_ylabel('Yards Gained')
    plt.sca(ax)
    plt.xticks(rotation=45)

# 2. Quarter
if 'quarter' in unique_frames.columns:
    ax = axes[plot_idx]
    plot_idx += 1
    
    quarter_data = unique_frames.drop_nulls(subset=['quarter']).to_pandas()
    quarter_data.boxplot(column='yards_gained', by='quarter', ax=ax)
    ax.set_title('Yards Gained by Quarter')
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Yards Gained')

# 3. Down
if 'down' in unique_frames.columns:
    ax = axes[plot_idx]
    plot_idx += 1
    
    down_data = unique_frames.drop_nulls(subset=['down']).to_pandas()
    down_data.boxplot(column='yards_gained', by='down', ax=ax)
    ax.set_title('Yards Gained by Down')
    ax.set_xlabel('Down')
    ax.set_ylabel('Yards Gained')

# 4. Distance to Goal
if 'distanceToGoal' in unique_frames.columns:
    ax = axes[plot_idx]
    plot_idx += 1
    
    # Bin distance to goal
    binned = unique_frames.with_columns(
        dist_bin=pl.when(pl.col('distanceToGoal') <= 10).then(pl.lit('Red Zone (<10)'))
        .when(pl.col('distanceToGoal') <= 30).then(pl.lit('Near Goal (10-30)'))
        .when(pl.col('distanceToGoal') <= 70).then(pl.lit('Midfield (30-70)'))
        .otherwise(pl.lit('Own Territory (70+)'))
    ).drop_nulls()
    
    dist_data = binned.to_pandas()
    dist_data.boxplot(column='yards_gained', by='dist_bin', ax=ax)
    ax.set_title('Yards Gained by Distance to Goal')
    ax.set_xlabel('Field Position')
    ax.set_ylabel('Yards Gained')
    plt.sca(ax)
    plt.xticks(rotation=45)

# Hide unused subplots
for i in range(plot_idx, 4):
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('Game State Impact on Yards Gained', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Compute correlations for game state features
# Note: These are constant per frame, so we need to aggregate per frame first
game_state_corr = {}

# Get unique frames with their game state and yards gained
unique_frames = df_with_target.select(
    ['gameId', 'playId', 'mirrored', 'frameId', 'yards_gained'] + 
    [c for c in game_state_features if c in df_with_target.columns and c != 'gameClock']
).unique()

print(f"Analyzing {len(unique_frames):,} unique frames")

for feat in game_state_features:
    if feat in unique_frames.columns and feat != 'gameClock':
        try:
            # Drop nulls before correlation
            valid_data = unique_frames.select([feat, 'yards_gained']).drop_nulls()
            if len(valid_data) > 0:
                corr = valid_data.to_pandas().corr().iloc[0, 1]
                game_state_corr[feat] = corr
        except Exception as e:
            print(f"Could not compute correlation for {feat}: {e}")

# Sort by absolute correlation
sorted_game_corr = sorted(game_state_corr.items(), key=lambda x: abs(x[1]), reverse=True)

print("\nGame State Correlation with Yards Gained:")
print("=" * 60)
for feat, corr in sorted_game_corr:
    print(f"  {feat:25s}: {corr:+.4f}")

## 9. Game State Features

Explore game context features that provide situational information:
- `yardsToGo`: Distance needed for first down
- `down`: Current down (1-4)
- `quarter`: Which quarter (1-4)
- `distanceToGoal`: Distance to opponent's goal line
- `half_seconds_remaining`: Seconds remaining in current half
- `gameClock`: Time remaining in quarter (string format)

These features are constant across all 22 players in a frame and provide critical context about game situation.